# M2.2.e: 169-feature integration + LightGBM val AUC

Integrates all M2.2 sub-steps into a single pipeline and runs LightGBM.

Pipeline: cloud_coverage fix → ClusterNo merge → sort → value-change (120)
→ SavGol residual → dayofyear → downsampling → CV split → StandardScaler → LightGBM

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.signal import savgol_filter
import lightgbm as lgb

train = pd.read_csv("../data/raw/train_features.csv")
print(f"Loaded: {train.shape}")

# M2.2.0: cloud_coverage sentinel fix
train["cloud_coverage"] = train["cloud_coverage"].replace({255: 10})
print("cloud_coverage fix applied")

Loaded: (1749494, 57)
cloud_coverage fix applied


In [2]:
clusterno_df = pd.read_csv("../data/interim/clusterno.csv")
train = train.merge(clusterno_df, on="building_id", how="left")
nan_clusterno = train["ClusterNo"].isna().sum()
print(f"After ClusterNo merge: {train.shape}")
print(f"ClusterNo NaN: {nan_clusterno}")
assert nan_clusterno == 0, f"Unexpected ClusterNo NaN: {nan_clusterno}"

After ClusterNo merge: (1749494, 58)
ClusterNo NaN: 0


In [3]:
train = train.sort_values(["building_id", "timestamp"]).reset_index(drop=True)

shifts = (
    list(np.arange(-24, 0))
    + list(np.arange(1, 25))
    + list(np.arange(-168, -24, 24))
    + list(np.arange(48, 169, 24))
)

for n in shifts:
    train[f"lag_value_{n}"] = (
        train.groupby("building_id")["meter_reading"].shift(n) - train["meter_reading"]
    )
    train[f"lag_value_ratio_{n}"] = (
        train.groupby("building_id")["meter_reading"].shift(n) + 1
    ) / (train["meter_reading"] + 1)

lag_cols = [c for c in train.columns if c.startswith("lag_value_")]
print(f"Lag features: {len(lag_cols)}")
print(f"Train shape: {train.shape}")
assert len(lag_cols) == 120, f"Expected 120, got {len(lag_cols)}"

C:\Users\tonykuo\AppData\Local\Temp\ipykernel_26240\182555182.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[f"lag_value_{n}"] = (
C:\Users\tonykuo\AppData\Local\Temp\ipykernel_26240\182555182.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[f"lag_value_ratio_{n}"] = (


Lag features: 120
Train shape: (1749494, 178)


In [4]:
results = []
for bid in train["building_id"].unique():
    tmp = train[train["building_id"] == bid].copy()
    smoothed = savgol_filter(tmp["meter_reading"].ffill().bfill().fillna(0), 5, 3)
    tmp["Residual_savgol_w5p3"] = tmp["meter_reading"] - smoothed
    results.append(tmp)
train = pd.concat(results).sort_index()
print(f"After SavGol: {train.shape}")

After SavGol: (1749494, 179)


In [5]:
train["dayofyear"] = (
    pd.to_datetime(train["timestamp"]).dt.dayofyear
    + pd.to_datetime(train["timestamp"]).dt.hour / 24
)
print(f"After dayofyear: {train.shape}")

After dayofyear: (1749494, 180)


In [6]:
X_full = train.select_dtypes(
    include=["float", "int", "int64", "int32", "float64", "float32"]
)
drop_cols = ["anomaly", "wind_direction", "air_temperature_std_lag73"]
X_full = X_full.drop(columns=[c for c in drop_cols if c in X_full.columns])

print(f"Feature count: {X_full.shape[1]}")
print("Expected: 169")
assert X_full.shape[1] == 169, (
    f"Feature count mismatch: got {X_full.shape[1]}, expected 169"
)
print("Feature count = 169")

categories = {
    "raw_numeric": 0,
    "lag_value_diff": 0,
    "lag_value_ratio": 0,
    "ClusterNo": 0,
    "Residual_savgol_w5p3": 0,
    "dayofyear": 0,
}
for col in X_full.columns:
    if col.startswith("lag_value_ratio_"):
        categories["lag_value_ratio"] += 1
    elif col.startswith("lag_value_"):
        categories["lag_value_diff"] += 1
    elif col == "ClusterNo":
        categories["ClusterNo"] += 1
    elif col == "Residual_savgol_w5p3":
        categories["Residual_savgol_w5p3"] += 1
    elif col == "dayofyear":
        categories["dayofyear"] += 1
    else:
        categories["raw_numeric"] += 1

print()
print("Feature category breakdown:")
for cat, count in categories.items():
    print(f"  {cat}: {count}")
print(f"  Total: {sum(categories.values())}")

Feature count: 169
Expected: 169
Feature count = 169

Feature category breakdown:
  raw_numeric: 46
  lag_value_diff: 60
  lag_value_ratio: 60
  ClusterNo: 1
  Residual_savgol_w5p3: 1
  dayofyear: 1
  Total: 169


In [7]:
print("NaN rate per feature class:")
class_specs = [
    (
        "raw_numeric",
        lambda c: not c.startswith("lag_value_")
        and c not in ["ClusterNo", "Residual_savgol_w5p3", "dayofyear"],
    ),
    (
        "lag_value_diff",
        lambda c: c.startswith("lag_value_") and not c.startswith("lag_value_ratio_"),
    ),
    ("lag_value_ratio", lambda c: c.startswith("lag_value_ratio_")),
    ("ClusterNo", lambda c: c == "ClusterNo"),
    ("Residual_savgol_w5p3", lambda c: c == "Residual_savgol_w5p3"),
    ("dayofyear", lambda c: c == "dayofyear"),
]
for name, fn in class_specs:
    cols = [c for c in X_full.columns if fn(c)]
    if not cols:
        continue
    total = len(cols) * len(X_full)
    nans = X_full[cols].isna().sum().sum()
    print(f"  {name:25s} {len(cols):4d} cols  NaN: {nans / total * 100:5.2f}%")

NaN rate per feature class:
  raw_numeric                 46 cols  NaN:  0.13%
  lag_value_diff              60 cols  NaN:  7.12%


  lag_value_ratio             60 cols  NaN:  7.12%
  ClusterNo                    1 cols  NaN:  0.00%
  Residual_savgol_w5p3         1 cols  NaN:  6.15%
  dayofyear                    1 cols  NaN:  0.00%


In [8]:
neg = train[train["anomaly"] == 0]
pos = train[train["anomaly"] == 1]
print(f"Pre-downsampling: {len(neg):,} normal + {len(pos):,} anomaly")

negs1 = neg.sample(n=pos.shape[0], random_state=10)
negs2 = neg.sample(n=pos.shape[0], random_state=20)
df_eq = pd.concat([negs1, pos, negs2, pos], axis=0).reset_index(drop=True)

print(f"Post-downsampling: {df_eq.shape[0]:,} rows")
print(
    f"Class balance: {(df_eq['anomaly'] == 0).sum():,} : {(df_eq['anomaly'] == 1).sum():,}"
)

Pre-downsampling: 1,712,198 normal + 37,296 anomaly


Post-downsampling: 149,184 rows
Class balance: 74,592 : 74,592


In [9]:
X_eq = df_eq.select_dtypes(include=["float", "int"])
X_eq = X_eq.drop(columns=[c for c in drop_cols if c in X_eq.columns])
print(f"X_eq features: {X_eq.shape[1]} (expected 169)")
assert X_eq.shape[1] == 169

train_mask = df_eq["building_id"] % 5 < 4
val_mask = df_eq["building_id"] % 5 == 4

X_train = X_eq[train_mask]
y_train = df_eq.loc[train_mask, "anomaly"]
X_val = X_eq[val_mask]
y_val = df_eq.loc[val_mask, "anomaly"]

print(f"Train: {X_train.shape}, {df_eq[train_mask]['building_id'].nunique()} buildings")
print(f"Val: {X_val.shape}, {df_eq[val_mask]['building_id'].nunique()} buildings")

X_eq features: 169 (expected 169)
Train: (119613, 169), 162 buildings
Val: (29571, 169), 38 buildings


In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
print(f"X_train_scaled: {X_train_scaled.shape}, NaN={np.isnan(X_train_scaled).sum():,}")
print(f"X_val_scaled:   {X_val_scaled.shape}, NaN={np.isnan(X_val_scaled).sum():,}")

X_train_scaled: (119613, 169), NaN=696,686
X_val_scaled:   (29571, 169), NaN=247,912


In [11]:
model = lgb.LGBMClassifier(n_estimators=100, verbose=-1, random_state=42)
model.fit(X_train_scaled, y_train)

preds = model.predict_proba(X_val_scaled)[:, 1]
val_auc = roc_auc_score(y_val, preds)

print(f"{'=' * 60}")
print(f"M2.2.e LightGBM val AUC: {val_auc:.4f}")
print(f"{'=' * 60}")

# M2.3 aliases
pred_lgb = preds
auc_lgb = val_auc
model_lgb = model

M2.2.e LightGBM val AUC: 0.9818


C:\Users\tonykuo\projects\lead-reproduction\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [12]:
baseline_auc = 0.8952  # M2.1
paper_target = 0.9849  # paper Table 2
delta_auc = val_auc - baseline_auc

print(f"M2.1 baseline (57 features):    {baseline_auc:.4f}")
print(f"M2.2.e (169 features):          {val_auc:.4f}")
print(f"delta_AUC vs M2.1:              {delta_auc:+.4f}")
print(f"Paper Table 2 target:           {paper_target:.4f}")
print(f"Gap vs paper:                   {(paper_target - val_auc) * 100:+.2f}%")
print()
print("Done when criteria:")
print(f"  val AUC >= 0.97:  {'pass' if val_auc >= 0.97 else 'FAIL'}")
print(f"  delta_AUC >= +0.04: {'pass' if delta_auc >= 0.04 else 'FAIL'}")

M2.1 baseline (57 features):    0.8952
M2.2.e (169 features):          0.9818
delta_AUC vs M2.1:              +0.0866
Paper Table 2 target:           0.9849
Gap vs paper:                   +0.31%

Done when criteria:
  val AUC >= 0.97:  pass
  delta_AUC >= +0.04: pass


In [13]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)
top10 = importances.nlargest(10)

print("Our top 10 importance:")
for i, (feat, imp) in enumerate(top10.items(), 1):
    print(f"  {i:2d}. {feat:35s} {imp:6.0f}")

paper_top10 = [
    "building_id",
    "lag_value_ratio_1",
    "lag_value_ratio_-1",
    "meter_reading",
    "dayofyear",
    "square_feet",
    "gte_building_id",
    "lag_value_ratio_-168",
    "lag_value_ratio_2",
    "gte_meter_primary_use",
]
print()
print("Paper Fig 5 top 10 (our name mapping):")
for i, feat in enumerate(paper_top10, 1):
    marker = "in ours" if feat in top10.index else "-"
    print(f"  {i:2d}. {feat:35s} {marker}")

overlap = set(paper_top10) & set(top10.index)
print(f"Overlap: {len(overlap)}/10")

Our top 10 importance:
   1. building_id                            206
   2. lag_value_ratio_1                      132
   3. meter_reading                          128
   4. lag_value_ratio_-1                     118
   5. dayofyear                              113
   6. Residual_savgol_w5p3                   105
   7. square_feet                             66
   8. lag_value_ratio_-168                    59
   9. lag_value_ratio_2                       56
  10. lag_value_ratio_168                     56

Paper Fig 5 top 10 (our name mapping):
   1. building_id                         in ours
   2. lag_value_ratio_1                   in ours
   3. lag_value_ratio_-1                  in ours
   4. meter_reading                       in ours
   5. dayofyear                           in ours
   6. square_feet                         in ours
   7. gte_building_id                     -
   8. lag_value_ratio_-168                in ours
   9. lag_value_ratio_2                   in ours
  1

In [14]:
import xgboost as xgb
import time

print("Training XGBoost (n_estimators=100)...")
t0 = time.time()
model_xgb = xgb.XGBClassifier(
    n_estimators=100, eval_metric="logloss", verbosity=0, random_state=42
)
model_xgb.fit(X_train_scaled, y_train)
elapsed_xgb = time.time() - t0
print(f"XGBoost trained in {elapsed_xgb:.1f}s")

pred_xgb = model_xgb.predict_proba(X_val_scaled)[:, 1]
auc_xgb = roc_auc_score(y_val, pred_xgb)
print(f"XGBoost val AUC: {auc_xgb:.4f}")

Training XGBoost (n_estimators=100)...


XGBoost trained in 2.0s
XGBoost val AUC: 0.9749


In [15]:
from catboost import CatBoostClassifier

print("Training CatBoost (iterations=1000, may take 5-15 min on CPU)...")
t0 = time.time()
model_cat = CatBoostClassifier(iterations=1000, verbose=False, random_seed=42)
model_cat.fit(X_train_scaled, y_train)
elapsed_cat = time.time() - t0
print(f"CatBoost trained in {elapsed_cat / 60:.1f} min")

pred_cat = model_cat.predict_proba(X_val_scaled)[:, 1]
auc_cat = roc_auc_score(y_val, pred_cat)
print(f"CatBoost val AUC: {auc_cat:.4f}")

Training CatBoost (iterations=1000, may take 5-15 min on CPU)...


CatBoost trained in 0.2 min
CatBoost val AUC: 0.9797


In [16]:
# CatBoost iteration verification
print("CatBoost configured iterations: 1000")
print(f"CatBoost actual tree_count_:    {model_cat.tree_count_}")
print(f"CatBoost best_iteration_:       {model_cat.best_iteration_}")

if model_cat.tree_count_ < 1000:
    print(f"\n⚠️  CatBoost only ran {model_cat.tree_count_} iterations (out of 1000)")
    print("   Likely cause: built-in overfitting detector triggered early stop")
    print("   This may NOT match buds-lab's exact run (which ran full 1000 iter)")
else:
    print("\n✓ CatBoost ran full 1000 iterations as configured")

CatBoost configured iterations: 1000
CatBoost actual tree_count_:    1000
CatBoost best_iteration_:       None

✓ CatBoost ran full 1000 iterations as configured


In [17]:
from sklearn.ensemble import HistGradientBoostingClassifier

# HistGBT does not support NaN in scaled data; fill with 0
X_train_filled = np.nan_to_num(X_train_scaled, nan=0)
X_val_filled = np.nan_to_num(X_val_scaled, nan=0)

print("Training HistGBT (max_iter=100)...")
t0 = time.time()
model_hist = HistGradientBoostingClassifier(max_iter=100, random_state=42)
model_hist.fit(X_train_filled, y_train)
elapsed_hist = time.time() - t0
print(f"HistGBT trained in {elapsed_hist:.1f}s")

pred_hist = model_hist.predict_proba(X_val_filled)[:, 1]
auc_hist = roc_auc_score(y_val, pred_hist)
print(f"HistGBT val AUC: {auc_hist:.4f}")

Training HistGBT (max_iter=100)...


HistGBT trained in 1.8s
HistGBT val AUC: 0.9806


In [18]:
# Equal-weight ensemble (per paper section 2.3.4 + Eq.4)
pred_ensemble = (pred_lgb + pred_xgb + pred_cat + pred_hist) / 4
auc_ensemble = roc_auc_score(y_val, pred_ensemble)

SEP = "=" * 60
sep = "-" * 50

print()
print(SEP)
print("M2.3 Results vs Paper Table 2")
print(SEP)
print(f"{'Model':<20s} {'Ours':>8s}   {'Paper':>8s}   {'Gap':>8s}")
print(sep)
print(f"{'LightGBM':<20s} {auc_lgb:.4f}    0.9849    {(0.9849 - auc_lgb) * 100:+.2f}%")
print(f"{'XGBoost':<20s} {auc_xgb:.4f}    0.9840    {(0.9840 - auc_xgb) * 100:+.2f}%")
print(f"{'CatBoost':<20s} {auc_cat:.4f}    0.9857    {(0.9857 - auc_cat) * 100:+.2f}%")
print(f"{'HistGBT':<20s} {auc_hist:.4f}    0.9839    {(0.9839 - auc_hist) * 100:+.2f}%")
print(sep)
print(
    f"{'Ensemble':<20s} {auc_ensemble:.4f}    0.9866    {(0.9866 - auc_ensemble) * 100:+.2f}%"
)
print()

cat_gt_lgb = auc_cat > auc_lgb
lgb_gt_xgb = auc_lgb > auc_xgb
xgb_gt_hist = auc_xgb > auc_hist
ranking_ok = cat_gt_lgb and lgb_gt_xgb and xgb_gt_hist
ensemble_best = auc_ensemble > max(auc_lgb, auc_xgb, auc_cat, auc_hist)
ensemble_pass = auc_ensemble >= 0.97

print("Done when:")
print(f"  Paper ranking Cat>LGB>XGB>HistGBT: {ranking_ok}")
print(
    f"    cat>lgb={cat_gt_lgb} ({auc_cat:.4f}>{auc_lgb:.4f}), lgb>xgb={lgb_gt_xgb}, xgb>hist={xgb_gt_hist}"
)
print(f"  Ensemble > max(individual): {ensemble_best}")
print(f"  Ensemble >= 0.97: {ensemble_pass}")


M2.3 Results vs Paper Table 2
Model                    Ours      Paper        Gap
--------------------------------------------------
LightGBM             0.9818    0.9849    +0.31%
XGBoost              0.9749    0.9840    +0.91%
CatBoost             0.9797    0.9857    +0.60%
HistGBT              0.9806    0.9839    +0.33%
--------------------------------------------------
Ensemble             0.9830    0.9866    +0.36%

Done when:
  Paper ranking Cat>LGB>XGB>HistGBT: False
    cat>lgb=False (0.9797>0.9818), lgb>xgb=True, xgb>hist=False
  Ensemble > max(individual): True
  Ensemble >= 0.97: True


In [19]:
# Cross-model feature importance comparison
# HistGBT (sklearn 1.7) does not expose feature_importances_; skipped for it.
import pandas as pd

model_lgb_name = "LightGBM"
models_with_imp = {
    "LightGBM": model_lgb,
    "XGBoost": model_xgb,
    "CatBoost": model_cat,
}

print("Top 5 feature importance per model (split-count / gain-based):")
print(f"{'Model':<12s}  Top 5 features")
print("-" * 80)

all_top10 = {}
for name, mdl in models_with_imp.items():
    imp = pd.Series(mdl.feature_importances_, index=X_train.columns)
    top5 = imp.nlargest(5).index.tolist()
    all_top10[name] = set(imp.nlargest(10).index)
    print(f"{name:<12s}  {top5}")

print()
print("Note: HistGBT skipped — sklearn 1.7 HistGradientBoostingClassifier")
print("      does not expose feature_importances_.")
print()

common_3 = set.intersection(*all_top10.values())
print("Features in all 3 models' top 10 (LGB / XGB / CatBoost):")
print(f"  Common: {sorted(common_3)}")
print(f"  Count:  {len(common_3)}/10")

print()
print("LGB-only top 10 (not in XGB or CatBoost):")
lgb_only = all_top10["LightGBM"] - all_top10["XGBoost"] - all_top10["CatBoost"]
print(f"  {sorted(lgb_only)}")

Top 5 feature importance per model (split-count / gain-based):
Model         Top 5 features
--------------------------------------------------------------------------------
LightGBM      ['building_id', 'lag_value_ratio_1', 'meter_reading', 'lag_value_ratio_-1', 'dayofyear']
XGBoost       ['meter_reading', 'lag_value_ratio_168', 'Residual_savgol_w5p3', 'lag_value_ratio_1', 'lag_value_ratio_19']
CatBoost      ['meter_reading', 'Residual_savgol_w5p3', 'building_id', 'lag_value_ratio_1', 'lag_value_ratio_-1']

Note: HistGBT skipped — sklearn 1.7 HistGradientBoostingClassifier
      does not expose feature_importances_.

Features in all 3 models' top 10 (LGB / XGB / CatBoost):
  Common: ['Residual_savgol_w5p3', 'lag_value_ratio_-1', 'lag_value_ratio_-168', 'lag_value_ratio_1', 'lag_value_ratio_168', 'meter_reading']
  Count:  6/10

LGB-only top 10 (not in XGB or CatBoost):
  ['lag_value_ratio_2']
